In [ ]:
import pandas as pd
import pickle
import numpy as np

# 1. Configuration
MODEL_FILE = "C:\\Users\\berna\\Desktop\\padel\\VoleAI2.0\\ml\\models\\final_padel_model.pkl"

# The exact features your model expects (in this specific order)
EXPECTED_FEATURES = [
    'match_quality_sum', 
    'court_speed_index', 
    'diff_log_total_points', 
    'diff_points_change', 
    'diff_tournaments_played_together', 
    'diff_matches_last_14_days', 
    'diff_finals_conversion_rate', 
    'diff_season_win_pct', 
    'diff_avg_games_conceded_per_set', 
    'diff_tie_break_win_pct', 
    'diff_comeback_rate', 
    'diff_avg_height'
]

def load_model(filename):
    with open(filename, 'rb') as f:
        model = pickle.load(f)
    print(f"✅ Model loaded from {filename}")
    return model

def predict_match(model, match_data):
    """
    match_data: A dictionary containing the values for the 12 features.
    """
    # Convert dictionary to DataFrame (Model expects a DataFrame with named columns)
    input_df = pd.DataFrame([match_data])
    
    # Ensure columns are in the correct order
    input_df = input_df[EXPECTED_FEATURES]
    
    # Predict Probability
    # returns [prob_loss, prob_win] -> we want [1] for Team 1 Win probability
    probability = model.predict_proba(input_df)[0][1]
    
    return probability

# =========================================
# EXAMPLE USAGE
# =========================================

# 1. Load the model
trained_model = load_model(MODEL_FILE)

# 2. Define a new match
# You need to calculate these values for the new match you want to predict.
# (Example values taken from an average match in your dataset)
new_match = {
    'match_quality_sum': 18.5,             # e.g., Sum of player ratings
    'court_speed_index': 52.5,             # e.g., Speed of the court
    'diff_log_total_points': 0.5,          # Positive means Team 1 has more ranking points
    'diff_points_change': 100.0,           # Team 1 gained more points recently
    'diff_tournaments_played_together': 2, # Team 1 has played 2 more tournaments together
    'diff_matches_last_14_days': 1,        # Team 1 is slightly more active
    'diff_finals_conversion_rate': 0.1,    # Team 1 is better at winning finals
    'diff_season_win_pct': 5,           # Team 1 has 5% higher win rate this season
    'diff_avg_games_conceded_per_set': -0.2, # Team 1 concedes fewer games (Negative is good)
    'diff_tie_break_win_pct': 0.0,         # Equal tie-break performance
    'diff_comeback_rate': 0.1,             # Team 1 is better at comebacks
    'diff_avg_height': 2.0                 # Team 1 is taller by 2cm
}

# 3. Get Prediction
win_prob = predict_match(trained_model, new_match)

print("-" * 30)
print(f"🏆 Prediction Result")
print(f"Probability Team 1 Wins: {win_prob:.2%}")
print("-" * 30)

if win_prob > 0.5:
    print("👉 Model predicts: TEAM 1 WINS")
else:
    print("👉 Model predicts: TEAM 1 LOSES")

In [9]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier
import pickle
# ==========================================
# CONFIGURATION
# ==========================================
DATA_FILE = "C:\\Users\\berna\\Desktop\\padel\\VoleAI2.0\\predictive_model\\data\\exploitation\\model_data.csv"
OUTPUT_FILE = "C:\\Users\\berna\\Desktop\\padel\\VoleAI2.0\\predictive_model\\ml\\models\\voting_soft_model.pkl"

# The exact features your model uses
FEATURES = [
    'match_quality_sum', 'court_speed_index', 'diff_log_total_points', 
    'diff_points_change', 'diff_tournaments_played_together', 
    'diff_matches_last_14_days', 'diff_finals_conversion_rate', 
    'diff_season_win_pct', 'diff_avg_games_conceded_per_set', 
    'diff_tie_break_win_pct', 'diff_comeback_rate', 'diff_avg_height'
]
TARGET = 'target_team1_wins'

# ==========================================
with open(OUTPUT_FILE, 'rb') as f:
    model = pickle.load(f)
print(f"✅ Loaded model from {OUTPUT_FILE}")

# ==========================================
# 4. GENERATE HYPOTHETICAL MATCHUPS
# ==========================================
print("Generating scenarios from real tournament days...")

def get_players(team_slug):
    # Simple check to prevent players playing against themselves
    # Assumes format "name-surname-name-surname"
    return set(team_slug.split('-'))

hypothetical_matches = []

# Group data by Date and Round (e.g., "2025-12-12 Semi-Finals")
groups = df.groupby(['date'])

for date, group in groups:
    if len(group) < 2: continue # Need at least 2 matches to swap opponents
        
    teams_today = pd.concat([group['team1_slug'], group['team2_slug']]).unique()
    
    # Try every possible pair of teams playing that day
    for i in range(len(teams_today)):
        for j in range(i + 1, len(teams_today)):
            t1 = teams_today[i]
            t2 = teams_today[j]
            
            # Filter 1: Skip if they ACTUALLY played each other that day
            played_real = False
            for _, row in group.iterrows():
                if (row['team1_slug'] == t1 and row['team2_slug'] == t2) or \
                   (row['team1_slug'] == t2 and row['team2_slug'] == t1):
                    played_real = True
                    break
            if played_real: continue
                
            # Filter 2: Skip if same player is on both sides (impossible)
            if not get_players(t1).isdisjoint(get_players(t2)):
                continue
            
            # Construct Feature Row using Latent Stats
            row = {}
            idx1 = team_to_idx[t1]
            idx2 = team_to_idx[t2]
            
            for feat in diff_features:
                row[feat] = latent_stats[feat][idx1] - latent_stats[feat][idx2]
            
            for feat in sum_features:
                row[feat] = latent_stats[feat][idx1] + latent_stats[feat][idx2]
            
            # Use environmental features from the actual day
            row['court_speed_index'] = group['court_speed_index'].mean()
            
            # Add metadata
            row['date'] = date
            row['team1'] = t1
            row['team2'] = t2
            
            hypothetical_matches.append(row)

# ==========================================
# 5. PREDICT & SAVE
# ==========================================
if not hypothetical_matches:
    print("No valid hypothetical combinations found.")
else:
    pred_df = pd.DataFrame(hypothetical_matches)
    
    # Predict
    X_pred = pred_df[FEATURES]
    pred_df['win_prob_team1'] = model.predict_proba(X_pred)[:, 1]
    
    # Sort: Recent dates first, then closest matches (probs near 50%)
    pred_df['uncertainty'] = abs(pred_df['win_prob_team1'] - 0.5)
    pred_df = pred_df.sort_values(['date', 'uncertainty'], ascending=[False, True])
    
    # Select columns to display
    cols_out = ['date','team1', 'team2', 'win_prob_team1']
    
    print("\n" + "="*50)
    print("TOP 10 INTERESTING HYPOTHETICAL MATCHUPS")
    print("="*50)
    print(pred_df[cols_out].head(30).to_string(index=False))
    
    print(f"\n✅ Saved {len(pred_df)} matchups to {OUTPUT_FILE}")

✅ Loaded model from C:\Users\berna\Desktop\padel\VoleAI2.0\predictive_model\ml\models\voting_soft_model.pkl
Generating scenarios from real tournament days...


KeyboardInterrupt: 